In [ ]:
# read:  cleaned_articles.parquet, sentiment_model
# write: sentiment_scores.parquet
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
path = '/content/drive/MyDrive/nlp_final_project/'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [ ]:
df = pd.read_parquet(path + 'cleaned_articles.parquet')
print(f"Loaded {len(df)} articles")


Loaded 196629 articles


In [ ]:
print(df[['article_id', 'text_clean']].tail())

        article_id                                         text_clean
196624      196624  Ejada Systems signs MoU with to enhance AI sol...
196625      196625  UNDP and e& strengthen AI collaboration for su...
196626      196626  Harnessing AI to make energy poverty history: ...
196627      196627  Reddit will start charging some users for API ...
196628      196628  Could 'the next Scarlett Johansson or Natalie ...


In [ ]:
# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = AutoModelForSequenceClassification.from_pretrained(path + 'sentiment_model/checkpoint-412')
model = model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:
# label map
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

print(f"Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

Model loaded — 124,647,939 parameters


In [ ]:
print(df['text_clean'].iloc[0])
print(type(df['text_clean'].iloc[0]))
print(df['text_clean'].isna().sum())
print(df['text_clean'].str.strip().eq('').sum())

Bad Idea AI Price (BAD), Market Cap, Price Today & Chart History - BlockworksOpen menuBrandsnewsletterspodcastseventsroundtablestoken transparencyetf trackerpricesresearchanalyticshomepricesBad Idea AI price (BAD)Bad Idea AIBADLive Bad Idea AI price updates and the latest Bad Idea AI byBlockworks Research$ $0(0%)24h low$ high$ Cap1D7D1M3M1YYTDALLVSUSDUSDBTCBTC TradingView ChartLinearLogarithmicThe live Bad Idea AI price today is $ with a 24-hour trading volume of $ The table above accurately updates our BAD price in real time. The price of BAD is down since last hour, up since yesterday. The live market cap, measured by multiplying the number of coins by the current price is $ BAD has a circulating supply of coins and a max supply of Idea AI StatsWhat is the market cap of Bad Idea AI?The current market cap of Bad Idea AI is $ A high market capitalization implies that the asset is highly valued by the is the current trading activity of Bad Idea AI?Currently, of BAD were traded within 24

In [ ]:
# article tokens check
sample = df['text_clean'].sample(3000, random_state=42).reset_index(drop=True)
lengths = sample.apply(lambda x: len(tokenizer.encode(x, truncation=False)))

print(lengths.describe())
over_512 = (lengths > 512).mean() * 100
print(f"\nArticles exceeding 512 tokens: {over_512:.1f}%")

Token indices sequence length is longer than the specified maximum sequence length for this model (1265 > 512). Running this sequence through the model will result in indexing errors


count     3000.000000
mean      1707.786667
std       1257.597101
min         39.000000
25%        991.000000
50%       1430.000000
75%       2126.250000
max      23309.000000
Name: text_clean, dtype: float64

Articles exceeding 512 tokens: 92.0%


In [ ]:
def chunk_text(text, tokenizer, max_length=512, stride=128):

    tokens = tokenizer.encode(text, truncation=False, add_special_tokens=False)

    if len(tokens) <= max_length - 2:
        return [tokens]

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_length - 2, len(tokens))
        chunks.append(tokens[start:end])
        if end == len(tokens):
            break
        start += stride

        if len(chunks) >= 10:
            break

    return chunks


def score_chunks(chunks, tokenizer, model, device):
    all_probs = []

    for chunk in chunks:
        input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
        input_ids = torch.tensor([input_ids]).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # logits to probabilities
        probs = torch.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu().numpy()[0])

    # mean aggregation across chunks
    # mean_probs = np.mean(all_probs, axis=0)
    # predicted_class = np.argmax(mean_probs)

    # after reviewing results at this tep i changed my approach to a weighted mean instead since training dataset was neutral heavy
    weights = np.array([1/0.134, 1/0.614, 1/0.252])
    weights = weights / weights.sum()
    mean_probs = np.mean(all_probs, axis=0) * weights
    mean_probs = mean_probs / mean_probs.sum()
    predicted_class = np.argmax(mean_probs)

    return {
        'sentiment_label': id2label[predicted_class],
        'sentiment_score': float(mean_probs[predicted_class]),
        'prob_negative': float(mean_probs[0]),
        'prob_neutral': float(mean_probs[1]),
        'prob_positive': float(mean_probs[2]),
        'num_chunks': len(chunks)
    }


In [ ]:
# test_text = df['text_clean'].iloc[196625]
# test_chunks = chunk_text(test_text, tokenizer)
# test_result = score_chunks(test_chunks, tokenizer, model, device)

# print(f"Text preview: {test_text[:200]}")
# print(f"Number of chunks: {test_result['num_chunks']}")
# print(f"Sentiment: {test_result['sentiment_label']} ({test_result['sentiment_score']:.3f})")
# print(f"Probabilities — neg: {test_result['prob_negative']:.3f} | "
#       f"neu: {test_result['prob_neutral']:.3f} | "
#       f"pos: {test_result['prob_positive']:.3f}")

In [ ]:
# read checkpoint
import os
import pandas as pd

checkpoint_file = path + 'sentiment_checkpoint_latest.parquet'

if os.path.exists(checkpoint_file):
    completed_df = pd.read_parquet(checkpoint_file)
    completed_ids = set(completed_df['article_id'].tolist())
    print(f"Checkpoint found: {len(completed_df):,} articles already scored")
    df_remaining = df[~df['article_id'].isin(completed_ids)].copy()
    print(f"Remaining articles to process: {len(df_remaining):,}")

    results = completed_df.to_dict('records')

else:
    print("No checkpoint found")
    df_remaining = df.copy()
    results = []

Checkpoint found: 50,000 articles already scored
Remaining articles to process: 146,629


In [ ]:
# full corpus
# results = []
CHECKPOINT_EVERY = 10000

for i, row in tqdm(df_remaining.iterrows(), total=len(df_remaining)):
    try:
        chunks = chunk_text(row['text_clean'], tokenizer)
        result = score_chunks(chunks, tokenizer, model, device)
        result['article_id'] = row['article_id']
        results.append(result)
    except Exception as e:
        results.append({
            'article_id': row['article_id'],
            'sentiment_label': 'unknown',
            'sentiment_score': None,
            'prob_negative': None,
            'prob_neutral': None,
            'prob_positive': None,
            'num_chunks': 0
        })

    # checkpoint every 10K
    if len(results) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_parquet(
            path + f'sentiment_checkpoint_latest.parquet',
            index=False
        )
        print(f"Checkpoint saved at {i+1} articles")

  7%|▋         | 10001/146629 [18:07<4:34:17,  8.30it/s]

Checkpoint saved at 60000 articles


 14%|█▎        | 20002/146629 [36:11<3:09:23, 11.14it/s]

Checkpoint saved at 70000 articles


 20%|██        | 30001/146629 [54:18<3:16:45,  9.88it/s]

...

 89%|████████▊ | 130001/146629 [3:54:31<47:51,  5.79it/s]

Checkpoint saved at 180000 articles


 95%|█████████▌| 140001/146629 [4:12:13<17:52,  6.18it/s]

Checkpoint saved at 190000 articles


100%|██████████| 146629/146629 [4:23:53<00:00,  9.26it/s]


In [ ]:
files = [f for f in os.listdir(path) if 'sentiment' in f and 'parquet' in f]
for f in sorted(files):
    temp = pd.read_parquet(path + f)
    print(f"{f}: {len(temp):,} rows | "
          f"ids: {temp['article_id'].min()} - {temp['article_id'].max()}")

sentiment_checkpoint_latest.parquet: 140,000 rows | ids: 50000 - 189999
sentiment_scores.parquet: 146,629 rows | ids: 50000 - 196628


In [ ]:
missing_ids = set(range(0, 50000))

df_missing = df[df['article_id'].isin(missing_ids)].copy()
print(f"Articles to process: {len(df_missing):,}")

missing_results = []

for i, row in tqdm(df_missing.iterrows(), total=len(df_missing)):
    try:
        chunks = chunk_text(row['text_clean'], tokenizer)
        result = score_chunks(chunks, tokenizer, model, device)
        result['article_id'] = row['article_id']
        missing_results.append(result)
    except Exception as e:
        missing_results.append({
            'article_id': row['article_id'],
            'sentiment_label': 'unknown',
            'sentiment_score': None,
            'prob_negative': None,
            'prob_neutral': None,
            'prob_positive': None,
            'num_chunks': 0
        })

missing_df = pd.DataFrame(missing_results)
missing_df.to_parquet(path + 'sentiment_missing_0_49999.parquet', index=False)
print(f"Missing articles scored: {len(missing_df):,}")

Articles to process: 50,000


100%|██████████| 50000/50000 [1:31:17<00:00,  9.13it/s]


Missing articles scored: 50,000


In [ ]:
# load existing and missing
existing_df = pd.read_parquet(path + 'sentiment_scores.parquet')
missing_df = pd.read_parquet(path + 'sentiment_missing_0_49999.parquet')

# merge
sentiment_df = pd.concat(
    [existing_df, missing_df],
    ignore_index=True
).drop_duplicates(subset=['article_id']).sort_values('article_id').reset_index(drop=True)

print(f"Total articles: {len(sentiment_df):,}")
print(f"Match main df:  {len(sentiment_df) == len(df)}")
print(f"ID range: {sentiment_df['article_id'].min()} - {sentiment_df['article_id'].max()}")
print(f"\nSentiment distribution:")
print(sentiment_df['sentiment_label'].value_counts())
print(f"Average chunks: {sentiment_df['num_chunks'].mean():.2f}")

Total articles: 196,629
Match main df:  True
ID range: 0 - 196628

Sentiment distribution:
sentiment_label
neutral     135886
positive     54198
negative      6545
Name: count, dtype: int64
Average chunks: 7.52


In [ ]:
sentiment_df.to_parquet(path + 'sentiment_scores.parquet', index=False)

In [ ]:
# sentiment_df = pd.DataFrame(results)

# print(f"\nSentiment distribution:")
# print(sentiment_df['sentiment_label'].value_counts())
# print(f"\nAverage chunks per article: {sentiment_df['num_chunks'].mean():.2f}")
# print(f"Articles scored: {len(sentiment_df)}")

# # final
# sentiment_df.to_parquet(path + 'sentiment_scores.parquet', index=False)

# print(sentiment_df[['article_id', 'sentiment_label', 'sentiment_score']].head(10))


Sentiment distribution:
sentiment_label
neutral     101387
positive     40418
negative      4824
Name: count, dtype: int64

Average chunks per article: 7.49
Articles scored: 146629
   article_id sentiment_label  sentiment_score
0       50000         neutral         0.531280
1       50001         neutral         0.995881
2       50002        positive         0.988443
3       50003         neutral         0.894263
4       50004        positive         0.600326
5       50005        positive         0.846337
6       50006         neutral         0.993374
7       50007         neutral         0.560323
8       50008        positive         0.605749
9       50009         neutral         0.987579
